# Big Mart Sales Prediction
## Exploration, Training & Predictions

This notebook walks through the complete pipeline:
1. **Exploratory Data Analysis (EDA)** – understand the dataset
2. **Preprocessing** – impute, normalise, and encode features
3. **Model Training** – fit a Ridge Regression pipeline
4. **Evaluation** – RMSE and R² on a held-out validation set
5. **Predictions** – generate sales estimates for the test set

> **Prerequisites:** Place `Train.csv` and `test.csv` inside `data/` before running.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import sys

# Add the project root to the Python path so `src` modules can be imported
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ── Third-party ───────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ── Project modules ───────────────────────────────────────────────────────────
from src.data_processing import load_data, clean_data, TARGET_COLUMN
from src.features import build_preprocessor, split_data, get_feature_names
from src.model import build_pipeline, train_model, evaluate_model, save_model
from src.predict import load_and_predict

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

print("✅ All imports successful.")

---
## 1 · Load & Inspect Raw Data

In [ ]:
TRAIN_PATH = "../data/Train.csv"
TEST_PATH  = "../data/test.csv"

df_raw = load_data(TRAIN_PATH)
print(f"Shape: {df_raw.shape}")
df_raw.head()

In [ ]:
print("Columns:", df_raw.columns.tolist())
print("\nData types:")
print(df_raw.dtypes)
print("\nMissing values:")
print(df_raw.isnull().sum())

In [ ]:
df_raw.describe()

---
## 2 · Exploratory Data Analysis

### 2.1 Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_raw[TARGET_COLUMN].dropna(), bins=50, color="steelblue", edgecolor="white", linewidth=0.4)
axes[0].set_title("Item Outlet Sales – Distribution")
axes[0].set_xlabel("Sales")
axes[0].set_ylabel("Count")

axes[1].hist(np.log1p(df_raw[TARGET_COLUMN].dropna()), bins=50, color="coral", edgecolor="white", linewidth=0.4)
axes[1].set_title("Item Outlet Sales – Log Distribution")
axes[1].set_xlabel("log(1 + Sales)")

plt.tight_layout()
os.makedirs("../reports", exist_ok=True)
plt.savefig("../reports/target_distribution.png", bbox_inches="tight")
plt.show()
print("Plot saved to reports/target_distribution.png")

### 2.2 Item_Fat_Content – unique values (before cleaning)

In [ ]:
print("Unique Item_Fat_Content values:")
print(df_raw["Item_Fat_Content"].value_counts())

### 2.3 Sales by Outlet Type

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(
    data=df_raw,
    x="Outlet_Type",
    y=TARGET_COLUMN,
    palette="Set2",
    ax=ax,
)
ax.set_title("Sales Distribution by Outlet Type")
ax.set_xlabel("Outlet Type")
ax.set_ylabel("Item Outlet Sales")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("../reports/sales_by_outlet_type.png", bbox_inches="tight")
plt.show()

### 2.4 MRP vs Sales

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    df_raw["Item_MRP"],
    df_raw[TARGET_COLUMN],
    alpha=0.25,
    s=8,
    c="steelblue",
)
ax.set_title("Item MRP vs Item Outlet Sales")
ax.set_xlabel("Item MRP")
ax.set_ylabel("Sales")
plt.tight_layout()
plt.savefig("../reports/mrp_vs_sales.png", bbox_inches="tight")
plt.show()

### 2.5 Correlation heatmap (numeric features)

In [ ]:
numeric_cols = ["Item_Weight", "Item_Visibility", "Item_MRP", "Outlet_Establishment_Year", TARGET_COLUMN]
corr = df_raw[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=ax, linewidths=0.5)
ax.set_title("Numeric Feature Correlation")
plt.tight_layout()
plt.savefig("../reports/correlation_heatmap.png", bbox_inches="tight")
plt.show()

---
## 3 · Data Cleaning & Preprocessing

In [ ]:
df_train = clean_data(df_raw, is_train=True)
print(f"Rows after cleaning: {len(df_train):,}  (raw: {len(df_raw):,})")
print("\nItem_Fat_Content after normalisation:")
print(df_train["Item_Fat_Content"].value_counts())
print("\nOutlet_Size nulls:", df_train["Outlet_Size"].isnull().sum())
print("Item_Weight nulls:", df_train["Item_Weight"].isnull().sum())

### 3.1 Train / Validation Split

In [ ]:
X_train, X_val, y_train, y_val = split_data(df_train, validation_ratio=0.2, random_state=42)
print(f"Training rows  : {len(X_train):,}")
print(f"Validation rows: {len(X_val):,}")

---
## 4 · Model Training

In [ ]:
pipeline = build_pipeline(alpha=0.8)
print("Pipeline:")
print(pipeline)

In [ ]:
pipeline = train_model(pipeline, X_train, y_train)
print("✅ Model trained.")

---
## 5 · Evaluation

In [ ]:
metrics = evaluate_model(pipeline, X_val, y_val)
print(f"Validation RMSE : {metrics['rmse']:.2f}")
print(f"Validation R²   : {metrics['r2']:.4f}")
print(f"Validation rows : {metrics['val_count']:,}")
print("\nTop features by |coefficient|:")
for feat in metrics["important_features"]:
    print(f"  {feat['feature']:<55}  coef = {feat['coefficient']:+.4f}")

### 5.1 Actual vs Predicted plot

In [ ]:
import numpy as np
y_pred_val = np.maximum(0, pipeline.predict(X_val))

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_val, y_pred_val, alpha=0.25, s=10, c="steelblue")
lims = [0, max(y_val.max(), y_pred_val.max()) * 1.05]
ax.plot(lims, lims, "r--", linewidth=1, label="Perfect fit")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Actual Sales")
ax.set_ylabel("Predicted Sales")
ax.set_title(f"Actual vs Predicted  (R² = {metrics['r2']:.3f})")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/actual_vs_predicted.png", bbox_inches="tight")
plt.show()

### 5.2 Top feature coefficients

In [ ]:
top_features = metrics["important_features"]
names  = [f["feature"].replace("num__", "").replace("cat__", "") for f in top_features]
coeffs = [f["coefficient"] for f in top_features]

colours = ["steelblue" if c >= 0 else "coral" for c in coeffs]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(names, coeffs, color=colours, edgecolor="white", linewidth=0.4)
ax.axvline(0, color="grey", linewidth=0.8)
ax.set_title("Top 6 Feature Coefficients (Ridge Regression)")
ax.set_xlabel("Coefficient value")
plt.tight_layout()
plt.savefig("../reports/feature_coefficients.png", bbox_inches="tight")
plt.show()

---
## 6 · Save Model

In [ ]:
MODEL_PATH = "../models/ridge_pipeline.joblib"
save_model(pipeline, MODEL_PATH)

---
## 7 · Generate Predictions on Test Set

In [ ]:
results = load_and_predict(TEST_PATH, MODEL_PATH)
output = results[["Item_Identifier", "Outlet_Identifier", "Predicted_Sales"]]
print(f"Test predictions generated: {len(output):,} rows")
output.head(10)

In [ ]:
output.to_csv("../reports/predictions.csv", index=False)
print("✅ Predictions saved to reports/predictions.csv")
print(f"\nSummary stats:")
print(output["Predicted_Sales"].describe().round(2))

---
## 8 · Prediction Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(output["Predicted_Sales"], bins=60, color="steelblue", edgecolor="white", linewidth=0.4)
ax.set_title("Test Set – Predicted Sales Distribution")
ax.set_xlabel("Predicted Sales")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig("../reports/predicted_sales_distribution.png", bbox_inches="tight")
plt.show()